<a href="https://colab.research.google.com/github/ckCrimson/Field_Dynamic_System/blob/FD_FRAMEWORK_PACKAGE/Research_paper_systems_implementation/random_dice_system_implementation/Random_Dice_System_%E2%80%94_A_Worked_Example_in_the_Field_Dynamic_Systems_Framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Git hub Setup

In [1]:
!rm -rf Field_Dynamic_System  # optional: clean old copy
!git clone -b FD_FRAMEWORK_PACKAGE https://github.com/ckCrimson/Field_Dynamic_System.git

Cloning into 'Field_Dynamic_System'...
remote: Enumerating objects: 27797, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 27797 (delta 87), reused 108 (delta 41), pack-reused 27613 (from 2)
Receiving objects: 100% (27797/27797), 157.66 MiB | 13.61 MiB/s, done.
Resolving deltas: 100% (7884/7884), done.


In [2]:
import sys, pathlib, types

project_root = pathlib.Path("/content/Field_Dynamic_System")
src_root = project_root / "src"

# 1) Make sure src/ is on sys.path so Python can see the fds package
if str(src_root) not in sys.path:
    sys.path.append(str(src_root))

# 2) Create a fake top-level package called "src" that points at src_root
src_pkg = types.ModuleType("src")
src_pkg.__path__ = [str(src_root)]
sys.modules["src"] = src_pkg

# 3) Now try importing
from fds import *
print("FDS Framework imported successfully!")


FDS Framework imported successfully!


# Random Dice System

## Core

### State

In [51]:
from fds.core.fds_state.state import State
from fds.core.fds_state.state_space import DiscreteFiniteStatSpace
import numpy as np
from numpy.typing import NDArray
from dataclasses import dataclass

#### Random Dice State

In [72]:
class DiceState(State):
  state: NDArray[np.float64]

  def __post_init__(self):
        # 1. Make the array immutable (read-only)
        self.state.flags.writeable = False

  def __hash__(self):
        # 2. Hash the bytes of the array data
        # Note: We must explicitly tell Python how to hash this field
        return hash(self.state.tobytes())

  def __eq__(self, other):
        # 3. Define equality check
        if not isinstance(other, ImmutableVector):
            return NotImplemented
        return np.array_equal(self.state, other.state)

  def __str__(self):
    return f"{self.state}"


#### Random Dice State Space

In [73]:
class DiceStateSpace(DiscreteFiniteStatSpace):
  def __init__(self,states: list[DiceState]=None, current_state: DiceState=None, dice_number: int = 6):
    self.dice_number = dice_number
    if current_state is None:
      current_state = DiceState( np.zeros(dice_number) )
    if states is None:
      states =[]
      states.append(current_state)
    super().__init__(states,current_state)